In [1]:
!pip -q install -U chromadb sentence-transformers google-genai python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:

import os
import re
import math
import numpy as np
import pandas as pd
import chromadb
from dotenv import load_dotenv
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

try:
    from google import genai
except Exception:
    genai = None

pd.set_option("display.max_colwidth", 200)


In [3]:
DATASET_PATH_CANDIDATES = [
    "/content/all_courses_combined_pathfinder.csv",
    "/content/all_courses_combined_pathfinder (1).csv",
    "/mnt/data/all_courses_combined_pathfinder (1).csv",
    "/mnt/data/all_courses_combined_pathfinder.csv",
    "all_courses_combined_pathfinder.csv",
    "all_courses_combined_pathfinder (1).csv",
]

dataset_path = next((p for p in DATASET_PATH_CANDIDATES if os.path.exists(p)), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Could not find all_courses_combined_pathfinder.csv. "
        "Upload it to Colab /content, then rerun this cell."
    )

df = pd.read_csv(dataset_path)

df.columns = [str(c).strip() for c in df.columns]

required_cols = ["course_title", "skills", "description"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}. Found columns: {list(df.columns)}")

optional_text_cols = [
    "provider", "level", "url", "track", "prerequisites", "roadmaps", "source_dataset"
]
for col in optional_text_cols:
    if col not in df.columns:
        df[col] = ""

optional_numeric_cols = ["rating", "reviews", "quality_weight"]
for col in optional_numeric_cols:
    if col not in df.columns:
        df[col] = np.nan

text_cols = ["course_title", "provider", "level", "url", "description", "skills", "track", "prerequisites", "roadmaps", "source_dataset"]
for col in text_cols:
    df[col] = df[col].fillna("").astype(str).str.strip()

def normalize_level_value(x):
    t = "" if pd.isna(x) else str(x).strip().lower()
    if not t or t in ["nan", "none", "unknown", "not calibrated"]:
        return "Unknown"
    if t in ["beginner", "introductory", "foundation", "novice"]:
        return "Beginner"
    if t in ["intermediate", "conversant"]:
        return "Intermediate"
    if t in ["advanced", "expert"]:
        return "Advanced"
    if "all" in t:
        return "All Levels"
    return str(x).strip().title()

df["level"] = df["level"].apply(normalize_level_value)

def build_fallback_description(row):
    pieces = [
        row.get("course_title", ""),
        row.get("skills", ""),
        row.get("track", ""),
        row.get("prerequisites", ""),
        row.get("roadmaps", ""),
    ]
    cleaned = ["" if pd.isna(x) else str(x).strip() for x in pieces]
    return " covers ".join([x for x in cleaned if x])

df["description"] = df.apply(
    lambda row: row["description"] if row["description"].strip() else build_fallback_description(row),
    axis=1
)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce").fillna(0.0)
df["reviews"] = pd.to_numeric(df["reviews"], errors="coerce").fillna(0.0)
df["quality_weight"] = pd.to_numeric(df["quality_weight"], errors="coerce").fillna(0.82).clip(0.0, 1.0)

df = df[df["course_title"].str.strip() != ""].copy()

df = df.drop_duplicates(subset=["course_title", "provider"], keep="first").reset_index(drop=True)

print("Dataset path:", dataset_path)
print("Shape after cleanup:", df.shape)
display(df.head(3))
print("\nColumns:", list(df.columns))
print("\nMissing values after safety cleanup:")
print(df.isna().sum())
print("\nSource distribution:")
print(df["source_dataset"].value_counts(dropna=False))


Dataset path: /content/all_courses_combined_pathfinder (1).csv
Shape after cleanup: (3685, 13)


,course_title,provider,level,rating,reviews,url,description,skills,track,prerequisites,roadmaps,source_dataset,quality_weight
0,Create an FPS Weapon in Unity (Part 1 - Revolver),Coursera Project Network,Advanced,5.0,0.0,https://www.coursera.org/learn/create-fps-weapon-unity-revolver,"In this one-hour, project-based course, you'll learn how to set up a revolver for a first-person shooter. This project covers configuring a gun prefab, enabling your FPS player to pick up, hold, f...",gamification of learning project process .properties hold java annotation source lines of code e-nable teaching method parenting computer-science software-development,core_it,,,Coursera_IT_cleaned,1.0
1,Encryption And Decryption Using C++,Coursera Project Network,Beginner,5.0,0.0,https://www.coursera.org/learn/encryption-and-decrytion-using-cpp,"In this 2-hour long project-based course, you will (learn basics of cryptography, build basic encryption application). we will learn basics of encryption and decryption techniques and gain basic r...",project cryptography project mine algorithms linux security modules form-based authentication cipher perl software encryption i-deas computer-science software-development,core_it,,,Coursera_IT_cleaned,1.0
2,Compare time series predictions of COVID-19 deaths,Coursera Project Network,Beginner,5.0,0.0,https://www.coursera.org/learn/compare-time-series-predictions-of-covid19-deaths,"In this 2-hour long project-based course, you will learn how to preprocess time series data, visualize time series data and compare the time series predictions of 4 machine learning models.You wil...",python programming modeling analysis project time series a t.i.m.e. training machine learning death time series analysis data-science machine-learning,core_it,,,Coursera_IT_cleaned,1.0



Columns: ['course_title', 'provider', 'level', 'rating', 'reviews', 'url', 'description', 'skills', 'track', 'prerequisites', 'roadmaps', 'source_dataset', 'quality_weight']

Missing values after safety cleanup:
course_title      0
provider          0
level             0
rating            0
reviews           0
url               0
description       0
skills            0
track             0
prerequisites     0
roadmaps          0
source_dataset    0
quality_weight    0
dtype: int64

Source distribution:
source_dataset
courses_csv                           2211
Coursera_IT_cleaned                   1032
coursera_course_dataset_v2_no_null     442
Name: count, dtype: int64


In [4]:
def guess_column(df, possible_names):
    normalized = {c.lower().strip().replace(" ", "_"): c for c in df.columns}
    for name in possible_names:
        key = name.lower().strip().replace(" ", "_")
        if key in normalized:
            return normalized[key]
    for c in df.columns:
        c_norm = c.lower().strip().replace(" ", "_")
        for p in possible_names:
            p_norm = p.lower().strip().replace(" ", "_")
            if p_norm in c_norm:
                return c
    return None

title_col = guess_column(df, ["course_title", "course title", "title", "name", "course_name"])
desc_col = guess_column(df, ["description", "course_description", "about", "summary"])
skills_col = guess_column(df, ["skills", "skill", "tags", "course_skills"])
level_col_original = guess_column(df, ["level", "difficulty", "difficulty_level"])
url_col = guess_column(df, ["url", "link", "course_url"])
provider_col = guess_column(df, ["provider", "platform", "partner", "organization", "university", "host"])
category_col = guess_column(df, ["track", "category", "domain", "subject", "it_label"])
prereq_col = guess_column(df, ["prerequisites", "prerequisite"])
roadmaps_col = guess_column(df, ["roadmaps", "roadmap"])
duration_col = guess_column(df, ["duration", "course_duration", "time"])
meta_col = guess_column(df, ["metadata", "meta"])
rating_col = guess_column(df, ["rating", "ratings", "score"])
reviews_col = guess_column(df, ["reviews", "review_count", "review counts", "review_counts"])
quality_weight_col = guess_column(df, ["quality_weight", "weight", "source_weight"])
source_dataset_col = guess_column(df, ["source_dataset", "source"])

for name, value in {
    "title_col": title_col,
    "desc_col": desc_col,
    "skills_col": skills_col,
    "level_col_original": level_col_original,
    "url_col": url_col,
    "provider_col": provider_col,
    "category_col": category_col,
    "prereq_col": prereq_col,
    "roadmaps_col": roadmaps_col,
    "duration_col": duration_col,
    "meta_col": meta_col,
    "rating_col": rating_col,
    "reviews_col": reviews_col,
    "quality_weight_col": quality_weight_col,
    "source_dataset_col": source_dataset_col,
}.items():
    print(f"{name}: {value}")

if title_col is None or skills_col is None or desc_col is None:
    raise ValueError("The notebook needs title, skills, and description columns to build embeddings.")


title_col: course_title
desc_col: description
skills_col: skills
level_col_original: level
url_col: url
provider_col: provider
category_col: track
prereq_col: prerequisites
roadmaps_col: roadmaps
duration_col: None
meta_col: None
rating_col: rating
reviews_col: reviews
quality_weight_col: quality_weight
source_dataset_col: source_dataset


In [5]:

def safe_str(x):
    return "" if pd.isna(x) else str(x)

def parse_level_from_metadata(text):
    if pd.isna(text):
        return None
    t = str(text).lower()
    if any(x in t for x in ["beginner", "foundation", "introductory", "novice"]):
        return "Beginner"
    if "intermediate" in t:
        return "Intermediate"
    if any(x in t for x in ["advanced", "expert"]):
        return "Advanced"
    if "all levels" in t:
        return "All Levels"
    return None

def parse_duration_hours(text):
    if pd.isna(text):
        return None
    t = str(text).lower().replace("–", "-").replace("—", "-").replace("approx.", "")

    m = re.search(r"(\d+(?:\.\d+)?)\s*(minute|minutes|min|mins)", t)
    if m:
        return float(m.group(1)) / 60.0

    m = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*(hour|hours|hr|hrs)", t)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2

    m = re.search(r"(\d+(?:\.\d+)?)\s*(hour|hours|hr|hrs)", t)
    if m:
        return float(m.group(1))

    m = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*(day|days)", t)
    if m:
        return ((float(m.group(1)) + float(m.group(2))) / 2) * 8

    m = re.search(r"(\d+(?:\.\d+)?)\s*(day|days)", t)
    if m:
        return float(m.group(1)) * 8

    m = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*(week|weeks)", t)
    if m:
        return ((float(m.group(1)) + float(m.group(2))) / 2) * 35

    m = re.search(r"(\d+(?:\.\d+)?)\s*(week|weeks)", t)
    if m:
        return float(m.group(1)) * 35

    m = re.search(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*(month|months)", t)
    if m:
        return ((float(m.group(1)) + float(m.group(2))) / 2) * 160

    m = re.search(r"(\d+(?:\.\d+)?)\s*(month|months)", t)
    if m:
        return float(m.group(1)) * 160

    return None

def bucket_duration(hours):
    if pd.isna(hours) or hours is None:
        return None
    if hours <= 10:
        return "Very Short"
    if hours <= 40:
        return "Short"
    if hours <= 120:
        return "Medium"
    return "Long"

def parse_review_count(value):
    if pd.isna(value):
        return 0.0
    t = str(value).lower().replace(",", "")
    m = re.search(r"\((\d+(?:\.\d+)?)\s*([km]?)\s*reviews?\)", t)
    if m:
        num = float(m.group(1))
        suffix = m.group(2)
        if suffix == "k":
            num *= 1_000
        elif suffix == "m":
            num *= 1_000_000
        return num
    m = re.search(r"(\d+(?:\.\d+)?)", t)
    return float(m.group(1)) if m else 0.0

def parse_certificate_type(text):
    if pd.isna(text):
        return None
    t = str(text).lower()
    mapping = [
        ("professional certificate", "Professional Certificate"),
        ("specialization", "Specialization"),
        ("guided project", "Guided Project"),
        ("course", "Course"),
        ("certificate", "Certificate"),
    ]
    for raw, norm in mapping:
        if raw in t:
            return norm
    return None

def split_skills(value):
    if pd.isna(value):
        return []
    raw = str(value)
    parts = re.split(r",|/|\||;", raw)
    cleaned = []
    for p in parts:
        p = re.sub(r"\s+", " ", p).strip()
        if p:
            cleaned.append(p)
    seen = set()
    out = []
    for s in cleaned:
        key = s.lower()
        if key not in seen:
            seen.add(key)
            out.append(s)
    return out

DOMAIN_KEYWORDS = {
    "data analysis": ["data analysis", "data analytics", "analytics", "spreadsheet", "excel", "tableau", "power bi", "business intelligence", "data visualization"],
    "machine learning": ["machine learning", "deep learning", "neural network", "tensorflow", "pytorch", "modeling", "ai", "artificial intelligence"],
    "data science": ["data science", "statistics", "statistical", "python", "r programming", "data mining"],
    "cybersecurity": ["cybersecurity", "security", "network security", "cryptography", "incident", "risk management", "siem"],
    "cloud": ["cloud", "aws", "azure", "gcp", "devops", "kubernetes", "docker"],
    "software development": ["java", "c++", "c#", "oop", "software engineering", "algorithms", "programming"],
    "web development": ["web", "html", "css", "javascript", "react", "frontend", "backend", "full stack"],
    "mobile development": ["android", "ios", "flutter", "react native", "mobile"],
    "project management": ["project management", "agile", "scrum", "stakeholder", "planning", "pmp"],
    "ux": ["ux", "ui", "design thinking", "wireframe", "prototype"],
}

def infer_domain_tags(row):
    blob = " ".join([
        safe_str(row.get(title_col, "")) if title_col else "",
        safe_str(row.get(desc_col, "")) if desc_col else "",
        safe_str(row.get(skills_col, "")) if skills_col else "",
        safe_str(row.get(category_col, "")) if category_col else "",
        safe_str(row.get(prereq_col, "")) if 'prereq_col' in globals() and prereq_col else "",
        safe_str(row.get(roadmaps_col, "")) if 'roadmaps_col' in globals() and roadmaps_col else "",
        safe_str(row.get(meta_col, "")) if meta_col else "",
    ]).lower()
    tags = []
    for domain, kws in DOMAIN_KEYWORDS.items():
        if any(kw in blob for kw in kws):
            tags.append(domain)
    return tags

df = df.copy()
df["course_id"] = df.index.astype(str)

if level_col_original:
    df["level"] = df[level_col_original].apply(lambda x: safe_str(x).strip() or None)
else:
    df["level"] = df[meta_col].apply(parse_level_from_metadata) if meta_col else None

if 'duration_col' in globals() and duration_col:
    df["duration_hours"] = df[duration_col].apply(parse_duration_hours)
elif meta_col:
    df["duration_hours"] = df[meta_col].apply(parse_duration_hours)
else:
    df["duration_hours"] = None
df["duration_bucket"] = df["duration_hours"].apply(bucket_duration)
df["certificate_type"] = df[meta_col].apply(parse_certificate_type) if meta_col else None
if meta_col:
    df["is_free"] = df[meta_col].fillna("").str.contains("free", case=False, na=False)
else:
    df["is_free"] = df.get("url", pd.Series("", index=df.index)).fillna("").str.contains("free", case=False, na=False)
df["rating_num"] = pd.to_numeric(df[rating_col], errors="coerce").fillna(0.0) if rating_col else 0.0
df["review_count_num"] = df[reviews_col].apply(parse_review_count) if reviews_col else 0.0
df["source_quality_weight"] = pd.to_numeric(df[quality_weight_col], errors="coerce").fillna(1.0) if quality_weight_col else 1.0
df["skill_list"] = df[skills_col].apply(split_skills) if skills_col else [[] for _ in range(len(df))]
df["skill_text_normalized"] = df["skill_list"].apply(lambda xs: " | ".join(x.lower() for x in xs))
df["domain_tags"] = df.apply(infer_domain_tags, axis=1)
df["domain_text"] = df["domain_tags"].apply(lambda xs: " | ".join(xs))

display_cols = [c for c in [
    title_col, meta_col, "level", "duration_hours", "duration_bucket",
    "certificate_type", "is_free", "rating_num", "review_count_num", "domain_text"
] if c in df.columns]
display(df[display_cols].head(10))



,course_title,level,duration_hours,duration_bucket,certificate_type,is_free,rating_num,review_count_num,domain_text
0,Create an FPS Weapon in Unity (Part 1 - Revolver),Advanced,None,None,None,False,5.0,0.0,machine learning | software development | ux
1,Encryption And Decryption Using C++,Beginner,None,None,None,False,5.0,0.0,machine learning | cybersecurity | software development | ux
2,Compare time series predictions of COVID-19 deaths,Beginner,None,None,None,False,5.0,0.0,machine learning | data science | software development
3,JavaScript Strings: Properties and Methods,Beginner,None,None,None,False,5.0,0.0,data science | software development | web development | mobile development
4,"Java Inheritance, Composition and Aggregation",Beginner,None,None,None,False,5.0,0.0,machine learning | data science | software development
5,Run a Sparkline Trend Analysis in Google Sheets,Beginner,None,None,None,False,5.0,0.0,data analysis | ux
6,Build CRUD REST API in Django,Intermediate,None,None,None,False,5.0,0.0,machine learning | software development | web development | ux
7,Artificial Intelligence Ethics in Action,Beginner,None,None,None,False,5.0,0.0,machine learning | software development
8,Design Computing: 3D Modeling in Rhinoceros with Python/Rhinoscript,Advanced,None,None,None,False,5.0,0.0,machine learning | data science | software development | ux
9,A Geometrical Approach to Genome Analysis: Skew & Z-Curve,Beginner,None,None,None,False,5.0,0.0,data science


In [6]:

def build_doc_text(row):
    parts = []
    if title_col:
        parts.append(f"Title: {safe_str(row.get(title_col))}")
    if desc_col:
        parts.append(f"Description: {safe_str(row.get(desc_col))}")
    if skills_col:
        parts.append(f"Skills: {safe_str(row.get(skills_col))}")
    if category_col:
        parts.append(f"Track/Category: {safe_str(row.get(category_col))}")
    if 'prereq_col' in globals() and prereq_col:
        parts.append(f"Prerequisites: {safe_str(row.get(prereq_col))}")
    if 'roadmaps_col' in globals() and roadmaps_col:
        parts.append(f"Roadmaps: {safe_str(row.get(roadmaps_col))}")
    if "level" in row and pd.notna(row.get("level")):
        parts.append(f"Level: {safe_str(row.get('level'))}")
    if "duration_hours" in row and pd.notna(row.get("duration_hours")):
        parts.append(f"Estimated duration hours: {row.get('duration_hours'):.1f}")
    if "certificate_type" in row and pd.notna(row.get("certificate_type")):
        parts.append(f"Credential type: {safe_str(row.get('certificate_type'))}")
    if "domain_text" in row and safe_str(row.get("domain_text")):
        parts.append(f"Domains: {safe_str(row.get('domain_text'))}")
    if provider_col:
        parts.append(f"Provider: {safe_str(row.get(provider_col))}")
    if meta_col:
        parts.append(f"Metadata: {safe_str(row.get(meta_col))}")
    return "\n".join([p for p in parts if p and p.strip()])

df["doc_text"] = df.apply(build_doc_text, axis=1)
display(df[[title_col, "doc_text"]].head(2))


,course_title,doc_text
0,Create an FPS Weapon in Unity (Part 1 - Revolver),"Title: Create an FPS Weapon in Unity (Part 1 - Revolver)\nDescription: In this one-hour, project-based course, you'll learn how to set up a revolver for a first-person shooter. This project covers..."
1,Encryption And Decryption Using C++,"Title: Encryption And Decryption Using C++\nDescription: In this 2-hour long project-based course, you will (learn basics of cryptography, build basic encryption application). we will learn basics..."


In [7]:

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
collection_name = "pathfinder_combined_courses"


def initialize_rag_index(force: bool = False):
    global model, embeddings, chroma_client, collection

    if (not force
        and "model" in globals()
        and "embeddings" in globals()
        and "collection" in globals()):
        return model, embeddings, collection

    print("Loading embedding model and building Chroma index...")
    model = SentenceTransformer(EMBED_MODEL_NAME)

    texts = df["doc_text"].fillna("").tolist()
    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass

    collection = chroma_client.create_collection(name=collection_name)

    metadatas = []
    for _, row in df.iterrows():
        metadatas.append({
            "course_id": safe_str(row.get("course_id")),
            "level": safe_str(row.get("level")) or "Unknown",
            "duration_hours": float(row.get("duration_hours")) if pd.notna(row.get("duration_hours")) else -1.0,
            "is_free": bool(row.get("is_free")) if "is_free" in row else False,
            "certificate_type": safe_str(row.get("certificate_type")) or "Unknown",
        })

    collection.add(
        ids=[str(i) for i in range(len(df))],
        documents=df["doc_text"].fillna("").tolist(),
        embeddings=embeddings.tolist(),
        metadatas=metadatas,
    )

    print(f"RAG index ready: {len(df)} courses indexed.")
    return model, embeddings, collection


initialize_rag_index(force=True)


Loading embedding model and building Chroma index...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/116 [00:00<?, ?it/s]

RAG index ready: 3685 courses indexed.


(SentenceTransformer(
   (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
   (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
   (2): Normalize({})
 ),
 array([[-0.01730511,  0.05262148, -0.04482052, ...,  0.01199584,
         -0.00666934,  0.01591864],
        [-0.04294406,  0.01450277, -0.00788247, ..., -0.03182651,
         -0.07526078,  0.00993979],
        [-0.02939599, -0.03897752, -0.03565063, ..., -0.03213335,
         -0.06607538,  0.05572226],
        ...,
        [-0.01097484,  0.00128952, -0.04873199, ...,  0.07947433,
          0.04514325, -0.07277383],
        [-0.02445357,  0.02976959, -0.05694608, ..., -0.04489675,
         -0.08922181, -0.02054781],
        [-0.0569527 , -0.03502893,  0.02841702, ...,  0.00112424,
         -0.06186144,  0.04210432]], dt

In [8]:

SKILL_SYNONYMS = {
    "python": ["python"],
    "sql": ["sql", "structured query language"],
    "excel": ["excel", "spreadsheet"],
    "power bi": ["power bi"],
    "tableau": ["tableau"],
    "machine learning": ["machine learning", "ml"],
    "deep learning": ["deep learning"],
    "statistics": ["statistics", "statistical"],
    "data analysis": ["data analysis", "data analytics", "analytics"],
    "nlp": ["nlp", "natural language processing"],
    "cloud": ["cloud", "aws", "azure", "gcp"],
    "cybersecurity": ["cybersecurity", "security", "network security"],
    "project management": ["project management", "agile", "scrum"],
    "java": ["java"],
    "c++": ["c++"],
    "react": ["react"],
    "flutter": ["flutter"],
}

def extract_constraints(query: str):
    q = query.lower()
    constraints = {
        "level": None,
        "max_hours": None,
        "must_free": False,
        "skill_keywords": [],
        "domain_keywords": [],
        "certificate_type": None,
        "practical_required": False,
    }

    if any(x in q for x in ["beginner", "introductory", "foundation"]):
        constraints["level"] = "Beginner"
    elif "intermediate" in q:
        constraints["level"] = "Intermediate"
    elif any(x in q for x in ["advanced", "expert"]):
        constraints["level"] = "Advanced"

    hour_match = re.search(r"(under|less than|<=|<)\s*(\d+(?:\.\d+)?)\s*(hour|hours|hr|hrs)", q)
    if hour_match:
        constraints["max_hours"] = float(hour_match.group(2))
    else:
        quick_hour = re.search(r"(\d+(?:\.\d+)?)\s*(hour|hours|hr|hrs)", q)
        if quick_hour and any(x in q for x in ["short", "quick", "fast"]):
            constraints["max_hours"] = float(quick_hour.group(1))
        elif any(x in q for x in ["short", "quick", "fast"]):
            constraints["max_hours"] = 40.0

    if "free" in q:
        constraints["must_free"] = True

    if "professional certificate" in q:
        constraints["certificate_type"] = "Professional Certificate"
    elif "specialization" in q:
        constraints["certificate_type"] = "Specialization"
    elif "guided project" in q:
        constraints["certificate_type"] = "Guided Project"
    elif "certificate" in q:
        constraints["certificate_type"] = "Certificate"

    if any(x in q for x in ["project", "hands-on", "handson", "practical", "capstone"]):
        constraints["practical_required"] = True

    skill_hits = []
    for canonical, synonyms in SKILL_SYNONYMS.items():
        if any(s in q for s in synonyms):
            skill_hits.append(canonical)
    constraints["skill_keywords"] = sorted(set(skill_hits))

    domain_hits = []
    for domain, kws in DOMAIN_KEYWORDS.items():
        if domain in q or any(kw in q for kw in kws):
            domain_hits.append(domain)
    constraints["domain_keywords"] = sorted(set(domain_hits))

    return constraints

extract_constraints("short beginner hands-on python course under 10 hours with certificate")


{'level': 'Beginner',
 'max_hours': 10.0,
 'must_free': False,
 'skill_keywords': ['python'],
 'domain_keywords': ['data science'],
 'certificate_type': 'Certificate',
 'practical_required': True}

In [9]:

def filter_by_metadata(df: pd.DataFrame, constraints: dict):
    filtered = df.copy()

    if constraints.get("level"):
        target_level = constraints["level"].lower()
        filtered = filtered[
            filtered["level"].fillna("").str.lower().isin([target_level, "all levels"])
        ]

    if constraints.get("max_hours") is not None:
        filtered = filtered[filtered["duration_hours"].notna()]
        filtered = filtered[filtered["duration_hours"] <= constraints["max_hours"]]

    if constraints.get("must_free"):
        filtered = filtered[filtered["is_free"] == True]

    if constraints.get("certificate_type"):
        filtered = filtered[
            filtered["certificate_type"].fillna("").str.lower().str.contains(
                re.escape(constraints["certificate_type"].lower()), na=False
            )
        ]

    skill_keywords = constraints.get("skill_keywords", [])
    if skill_keywords:
        mask = pd.Series(False, index=filtered.index)
        skill_series = filtered["skill_text_normalized"].fillna("")
        title_series = filtered[title_col].fillna("").str.lower() if title_col else pd.Series("", index=filtered.index)
        for kw in skill_keywords:
            mask = mask | skill_series.str.contains(re.escape(kw), case=False, na=False)
            mask = mask | title_series.str.contains(re.escape(kw), case=False, na=False)
        filtered = filtered[mask]

    domain_keywords = constraints.get("domain_keywords", [])
    if domain_keywords:
        mask = pd.Series(False, index=filtered.index)
        domain_series = filtered["domain_text"].fillna("").str.lower()
        for kw in domain_keywords:
            mask = mask | domain_series.str.contains(re.escape(kw), case=False, na=False)
        filtered = filtered[mask]

    return filtered

test_constraints = extract_constraints("short beginner hands-on python course under 10 hours")
print(test_constraints)
print("Filtered rows:", len(filter_by_metadata(df, test_constraints)))


{'level': 'Beginner', 'max_hours': 10.0, 'must_free': False, 'skill_keywords': ['python'], 'domain_keywords': ['data science'], 'certificate_type': None, 'practical_required': True}
Filtered rows: 0


In [10]:
def semantic_search(query, top_k=5):
    initialize_rag_index(force=False)

    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    res = collection.query(query_embeddings=q_emb.tolist(), n_results=top_k)

    idxs = [int(x) for x in res["ids"][0]]
    distances = res["distances"][0]
    scores = [1 - d for d in distances]

    out = df.iloc[idxs].copy()
    out["semantic_score"] = scores
    return out.reset_index(drop=True)

semantic_search("learn data analysis for internship", top_k=5)


,course_title,provider,level,rating,reviews,url,description,skills,track,prerequisites,...,is_free,rating_num,review_count_num,source_quality_weight,skill_list,skill_text_normalized,domain_tags,domain_text,doc_text,semantic_score
0,Your Guide to Learn Data Analysis,almentor,Beginner,0.0,0.0,https://www.almentor.net/en/courses/Your-Guide-to-Learn-Data-Analysis,"Taught by Nermin Elgrawany. Having chunks of data is one thing, and using that data to empower strategic business decisions is a whole another story. Simply because, if you don’t know how to explo...",data_analysis|business_intelligence There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this c...,data_analysis|business_intelligence,There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this course.,...,False,0.0,0.0,0.82,"[data_analysis, business_intelligence There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this...",data_analysis | business_intelligence there are no requirements for this course. your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this...,"[data analysis, software development, ux]",data analysis | software development | ux,"Title: Your Guide to Learn Data Analysis\nDescription: Taught by Nermin Elgrawany. Having chunks of data is one thing, and using that data to empower strategic business decisions is a whole anothe...",0.378812
1,The Analysis Fundamentals Course,almentor,Beginner,0.0,0.0,https://www.almentor.net/en/courses/The-Analysis-Fundamentals-Course,Taught by Mohamed AbdelMegeed. This course offers a comprehensive understanding of Excel for effective data analysis. You'll learn how big companies leverage data for strategic decisions and gain ...,"data_analysis|business_intelligence 1- a computer or laptop with Microsoft Windows. 2- Microsoft Excel 2016, Excel 2019, or Microsoft 365. 3- No Prior Excel Knowledge ...",data_analysis|business_intelligence,"1- a computer or laptop with Microsoft Windows. 2- Microsoft Excel 2016, Excel 2019, or Microsoft 365. 3- No Prior Excel Knowledge Required.",...,False,0.0,0.0,0.82,"[data_analysis, business_intelligence 1- a computer or laptop with Microsoft Windows. 2- Microsoft Excel 2016, Excel 2019, or Microsoft 365. 3- No Prior Excel Knowledge Required.]",data_analysis | business_intelligence 1- a computer or laptop with microsoft windows. 2- microsoft excel 2016 | excel 2019 | or microsoft 365. 3- no prior excel knowledge required.,"[data analysis, machine learning, data science, ux]",data analysis | machine learning | data science | ux,Title: The Analysis Fundamentals Course\nDescription: Taught by Mohamed AbdelMegeed. This course offers a comprehensive understanding of Excel for effective data analysis. You'll learn how big com...,0.312396
2,Advanced Data Analytics,almentor,Beginner,0.0,0.0,https://www.almentor.net/en/courses/Advanced-Data-Analytics,"Taught by Mohamed AbdelMegeed. Data analytics is the science of analyzing raw data to make conclusions about that information. It helps businesses in mastering performance optimization, performing...",data_analysis|business_intelligence There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this c...,data_analysis|business_intelligence,There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this course.,...,False,0.0,0.0,0.82,"[data_analysis, business_intelligence There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit

In [11]:
def retrieve_courses(query, top_k = 15, fetch_k=50):
    initialize_rag_index(force=False)

    constraints = extract_constraints(query)
    filtered_df = filter_by_metadata(df, constraints)

    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]

    if not filtered_df.empty:
        candidate_indices = filtered_df.index.to_numpy()
        candidate_embs = embeddings[candidate_indices]
        semantic_scores = np.dot(candidate_embs, q_emb)
        order = np.argsort(-semantic_scores)[:max(top_k, fetch_k)]
        chosen_indices = candidate_indices[order]
        out = df.iloc[chosen_indices].copy()
        out["semantic_score"] = semantic_scores[order]
        out["filter_applied"] = True
        return out.reset_index(drop=True)

    res = collection.query(query_embeddings=[q_emb.tolist()], n_results=max(top_k, fetch_k))
    idxs = [int(x) for x in res["ids"][0]]
    distances = res["distances"][0]
    scores = [1 - d for d in distances]

    out = df.iloc[idxs].copy()
    out["semantic_score"] = scores
    out["filter_applied"] = False
    return out.reset_index(drop=True)


In [12]:

def min_max_normalize(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    s_min, s_max = s.min(), s.max()
    if s_max == s_min:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s_min) / (s_max - s_min)

def compute_keyword_match_score(results_df, constraints):
    score = pd.Series(0.0, index=results_df.index)

    text_fields = [
        results_df[title_col].fillna("").str.lower() if title_col else pd.Series("", index=results_df.index),
        results_df["doc_text"].fillna("").str.lower(),
        results_df["skill_text_normalized"].fillna("").str.lower(),
        results_df["domain_text"].fillna("").str.lower(),
    ]
    combined = text_fields[0]
    for s in text_fields[1:]:
        combined = combined + " " + s

    for kw in constraints.get("skill_keywords", []):
        score += combined.str.contains(re.escape(kw), na=False).astype(float)

    for kw in constraints.get("domain_keywords", []):
        score += combined.str.contains(re.escape(kw), na=False).astype(float)

    practical_terms = ["project", "hands-on", "handson", "practical", "capstone"]
    if constraints.get("practical_required"):
        score += combined.apply(lambda x: 1.0 if any(t in x for t in practical_terms) else 0.0)

    return min_max_normalize(score)

def compute_constraint_match_score(results_df, constraints):
    score = pd.Series(0.0, index=results_df.index)

    if constraints.get("level"):
        target = constraints["level"].lower()
        score += results_df["level"].fillna("").str.lower().isin([target, "all levels"]).astype(float)

    if constraints.get("max_hours") is not None:
        score += (
            results_df["duration_hours"].fillna(10**9) <= constraints["max_hours"]
        ).astype(float)

    if constraints.get("must_free"):
        score += results_df["is_free"].fillna(False).astype(float)

    if constraints.get("certificate_type"):
        score += results_df["certificate_type"].fillna("").str.lower().str.contains(
            re.escape(constraints["certificate_type"].lower()), na=False
        ).astype(float)

    if constraints.get("practical_required"):
        practical_blob = (
            results_df["doc_text"].fillna("").str.lower() + " " +
            results_df["skill_text_normalized"].fillna("").str.lower()
        )
        score += practical_blob.apply(
            lambda x: 1.0 if any(t in x for t in ["project", "hands-on", "handson", "practical", "capstone"]) else 0.0
        )

    return min_max_normalize(score)

def rerank_results(results_df, query, sim_weight=0.45, rating_weight=0.15, review_weight=0.00,
                   keyword_weight=0.16, constraint_weight=0.08, quality_weight=0.16):
    ranked = results_df.copy()
    if "semantic_score" not in ranked.columns:
        raise ValueError("Results must contain 'semantic_score' before re-ranking.")

    constraints = extract_constraints(query)

    ranked["semantic_score_norm"] = min_max_normalize(ranked["semantic_score"])
    ranked["rating_score_norm"] = min_max_normalize(ranked["rating_num"])
    ranked["review_score_norm"] = 0.0
    ranked["keyword_match_norm"] = compute_keyword_match_score(ranked, constraints)
    ranked["constraint_match_norm"] = compute_constraint_match_score(ranked, constraints)
    ranked["quality_weight_norm"] = pd.to_numeric(
        ranked.get("source_quality_weight", 1.0), errors="coerce"
    ).fillna(1.0).clip(lower=0.0, upper=1.0)

    base_score = (
        sim_weight * ranked["semantic_score_norm"] +
        rating_weight * ranked["rating_score_norm"] +
        review_weight * ranked["review_score_norm"] +
        keyword_weight * ranked["keyword_match_norm"] +
        constraint_weight * ranked["constraint_match_norm"] +
        quality_weight * ranked["quality_weight_norm"]
    )

    ranked["final_score"] = base_score * (0.85 + 0.15 * ranked["quality_weight_norm"])

    ranked = ranked.sort_values(
        ["final_score", "constraint_match_norm", "keyword_match_norm", "semantic_score_norm"],
        ascending=False
    ).reset_index(drop=True)
    return ranked

ranked_demo = rerank_results(
    retrieve_courses("learn data analysis for internship", top_k = 15, fetch_k=50),
    query="learn data analysis for internship"
)
display(ranked_demo[[c for c in [
    title_col, "semantic_score", "rating_num", "review_count_num",
    "keyword_match_norm", "constraint_match_norm", "final_score"
] if c in ranked_demo.columns]].head(5))



,course_title,semantic_score,rating_num,review_count_num,keyword_match_norm,constraint_match_norm,final_score
0,Your Guide to Learn Data Analysis,0.689406,0.0,0.0,0.0,0.0,0.565508
1,Analyze Data to Answer Questions,0.626333,4.6,0.0,0.0,0.0,0.509154
2,Exploratory Data Analysis With Python and Pandas,0.582679,4.7,0.0,0.0,0.0,0.464807
3,Introduction to Data Analytics for Business,0.602415,4.6,0.0,0.0,0.0,0.446646
4,Data Science Using Microsoft Excel Business Intelligence (BI),0.641369,0.0,0.0,0.0,0.0,0.437599


In [13]:

def build_reason(row, constraints):
    reasons = []

    matched_skill_terms = []
    for kw in constraints.get("skill_keywords", []):
        blob = (
            safe_str(row.get(title_col, "")) + " " +
            safe_str(row.get("skill_text_normalized", "")) + " " +
            safe_str(row.get("doc_text", ""))
        ).lower()
        if kw in blob:
            matched_skill_terms.append(kw)

    matched_domains = []
    for kw in constraints.get("domain_keywords", []):
        if kw in safe_str(row.get("domain_text", "")).lower():
            matched_domains.append(kw)

    if matched_skill_terms:
        reasons.append("Matches skills/topics: " + ", ".join(matched_skill_terms[:4]))
    if matched_domains:
        reasons.append("Aligned domain: " + ", ".join(matched_domains[:3]))
    if constraints.get("level") and safe_str(row.get("level", "")).lower() in [constraints["level"].lower(), "all levels"]:
        reasons.append(f"Suitable for {safe_str(row.get('level'))}")
    if constraints.get("max_hours") is not None and pd.notna(row.get("duration_hours")):
        reasons.append(f"Fits time limit (~{row.get('duration_hours'):.1f} hrs)")
    if pd.notna(row.get("rating_num")) and row.get("rating_num", 0) > 0:
        reasons.append(f"Strong rating ({row.get('rating_num'):.1f})")
    if pd.notna(row.get("review_count_num")) and row.get("review_count_num", 0) > 0:
        reasons.append(f"Popular ({int(row.get('review_count_num')):,} reviews)")
    if constraints.get("practical_required"):
        practical_blob = (safe_str(row.get("doc_text", "")) + " " + safe_str(row.get("skill_text_normalized", ""))).lower()
        if any(t in practical_blob for t in ["project", "hands-on", "handson", "practical", "capstone"]):
            reasons.append("Has practical/project signal")

    if not reasons:
        reasons.append("Retrieved from the most relevant grounded course rows")
    return "; ".join(reasons[:3])

def recommend(goal_text, top_k = 15, fetch_k=50):
    constraints = extract_constraints(goal_text)
    retrieved = retrieve_courses(goal_text, top_k=max(top_k, fetch_k), fetch_k=fetch_k)
    ranked = rerank_results(retrieved, query=goal_text).head(top_k)

    print("GOAL:", goal_text)
    print("=" * 100)

    for rank, (_, row) in enumerate(ranked.iterrows(), start=1):
        title = safe_str(row.get(title_col, "N/A")) if title_col else "N/A"
        level_value = safe_str(row.get("level", "N/A"))
        provider = safe_str(row.get(provider_col, "N/A")) if provider_col else "N/A"
        url = safe_str(row.get(url_col, "N/A")) if url_col else "N/A"
        rating = row.get("rating_num", np.nan)
        duration = row.get("duration_hours", np.nan)
        cert = safe_str(row.get("certificate_type", "N/A"))
        ref = safe_str(row.get("course_id", "N/A"))

        print(f"{rank}) {title}")
        print(f"   Provider: {provider} | Level: {level_value} | Rating: {rating:.1f}" if pd.notna(rating) else f"   Provider: {provider} | Level: {level_value}")
        if pd.notna(duration):
            print(f"   Duration (hrs): {duration:.1f} | Credential: {cert}")
        else:
            print(f"   Credential: {cert}")
        print(f"   Why it matches: {build_reason(row, constraints)}")
        print(f"   Reference: course_id={ref}" + (f" | url={url}" if url and url != "N/A" else ""))
        print("-" * 100)

    return ranked

recommend("learn data analysis for internship", top_k = 15, fetch_k=50)


GOAL: learn data analysis for internship
1) Your Guide to Learn Data Analysis
   Provider: almentor | Level: Beginner | Rating: 0.0
   Credential: 
   Why it matches: Matches skills/topics: data analysis; Aligned domain: data analysis
   Reference: course_id=1598 | url=https://www.almentor.net/en/courses/Your-Guide-to-Learn-Data-Analysis
----------------------------------------------------------------------------------------------------
2) Analyze Data to Answer Questions
   Provider: Google | Level: Beginner | Rating: 4.6
   Credential: 
   Why it matches: Matches skills/topics: data analysis; Aligned domain: data analysis; Strong rating (4.6)
   Reference: course_id=1277
----------------------------------------------------------------------------------------------------
3) Exploratory Data Analysis With Python and Pandas
   Provider: Coursera Project Network | Level: Beginner | Rating: 4.7
   Credential: 
   Why it matches: Matches skills/topics: data analysis; Aligned domain: data a

,course_title,provider,level,rating,reviews,url,description,skills,track,prerequisites,...,doc_text,semantic_score,filter_applied,semantic_score_norm,rating_score_norm,review_score_norm,keyword_match_norm,constraint_match_norm,quality_weight_norm,final_score
0,Your Guide to Learn Data Analysis,almentor,Beginner,0.0,0.0,https://www.almentor.net/en/courses/Your-Guide-to-Learn-Data-Analysis,"Taught by Nermin Elgrawany. Having chunks of data is one thing, and using that data to empower strategic business decisions is a whole another story. Simply because, if you don’t know how to explo...",data_analysis|business_intelligence There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this c...,data_analysis|business_intelligence,There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this course.,...,"Title: Your Guide to Learn Data Analysis\nDescription: Taught by Nermin Elgrawany. Having chunks of data is one thing, and using that data to empower strategic business decisions is a whole anothe...",0.689406,True,1.000000,0.000000,0.0,0.0,0.0,0.82,0.565508
1,Analyze Data to Answer Questions,Google,Beginner,4.6,0.0,,"Analyze Data to Answer Questions covers data analysis, data management, microsoft excel, sql, spreadsheet software","data analysis, data management, microsoft excel, sql, spreadsheet software",semi_it,,...,"Title: Analyze Data to Answer Questions\nDescription: Analyze Data to Answer Questions covers data analysis, data management, microsoft excel, sql, spreadsheet software\nSkills: data analysis, dat...",0.626333,True,0.616435,0.958333,0.0,0.0,0.0,0.70,0.509154
2,Exploratory Data Analysis With Python and Pandas,Coursera Project Network,Beginner,4.7,0.0,https://www.coursera.org/learn/exploratory-data-analysis-python-pandas,"In this 2-hour long project-based course, you will learn how to perform Exploratory Data Analysis (EDA) in Python. You will use external Python packages such as Pandas, Numpy, Matplotlib, Seaborn ...",data analysis meta-analysis analysis exploratory data analysis pandas data visualization univariate analysis python programming project numpy data-science data-analysis,core_it,,...,"Title: Exploratory Data Analysis With Python and Pandas\nDescription: In this 2-hour long project-based course, you will learn how to perform Exploratory Data Analysis (EDA) in Python. You will us...",0.582679,True,0.350960,0.979167,0.0,0.0,0.0,1.00,0.464807
3,Introduction to Data Analytics for Business,University of Colorado Boulder,Advanced,4.6,0.0,https://www.coursera.org/learn/data-analytics-business,"This course will expose you to the data analytics practices executed in the business world. We will explore such key areas as the analytical process, how data is created, stored, accessed, and how...",analysis analytics sql business analysis relational database data analysis data model databases business analytics modeling data-science data-analysis,semi_it,,...,Title: Introduction to Data Analytics for Business\nDescription: This course will expose you to the data analytics practices executed in the business world. We will explore such key areas as the a...,0.602415,True,0.470983,0.958333,0.0,0.0,0.0,0.70,0.446646
4,Data Science Using Microsoft Excel Business Intelligence (BI),almentor,Beginner,0.0,0.0,https://www.almentor.net/en/courses/Data-Science-Using-Microsoft-Excel-Business-Intelligence-(BI),"Taught by Ashraf Elsheikh. We are now living in the age of big data. Data is constantly being collected, and an overwhelming amount of data exists. There is hence a growing need for people who can...",data_analysis|business_intelligence This course requires the installation of Microsoft Excel. Prior experience with Microsoft Excel is preferable.. Completing the Business Data Analysis using Exce...,data_analysis|business_intelligence,"This 

In [14]:


def evaluate_recommendation(goal, course_row):
    constraints = extract_constraints(goal)
    score = 0
    notes = []

    blob = (
        safe_str(course_row.get(title_col, "")) + " " +
        safe_str(course_row.get("doc_text", "")) + " " +
        safe_str(course_row.get("skill_text_normalized", "")) + " " +
        safe_str(course_row.get("domain_text", ""))
    ).lower()

    all_topics = constraints.get("skill_keywords", []) + constraints.get("domain_keywords", [])
    if all_topics and any(topic in blob for topic in all_topics):
        score += 1
        notes.append("Topic match")

    level_ok = True
    if constraints.get("level"):
        level_ok = safe_str(course_row.get("level", "")).lower() in [constraints["level"].lower(), "all levels"]

    time_ok = True
    if constraints.get("max_hours") is not None and pd.notna(course_row.get("duration_hours")):
        time_ok = float(course_row.get("duration_hours")) <= float(constraints["max_hours"])

    cert_ok = True
    if constraints.get("certificate_type"):
        cert_ok = constraints["certificate_type"].lower() in safe_str(course_row.get("certificate_type", "")).lower()

    free_ok = True
    if constraints.get("must_free"):
        free_ok = bool(course_row.get("is_free", False))

    if level_ok and time_ok and cert_ok and free_ok:
        score += 1
        notes.append("Constraint match")

    practical_terms = ["project", "hands-on", "handson", "practical", "capstone"]
    if constraints.get("practical_required"):
        if any(x in blob for x in practical_terms):
            score += 1
            notes.append("Practical signal")
    elif any(x in blob for x in practical_terms):
        score += 1
        notes.append("Practical signal")

    return score, "; ".join(notes)


def dcg_at_k(relevance_scores, k=15):
    rel = np.asarray(relevance_scores[:k], dtype=float)
    if rel.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rel.size + 2))
    return float(np.sum((2**rel - 1) / discounts))


def ndcg_at_k(relevance_scores, k=15):
    actual = dcg_at_k(relevance_scores, k=k)
    ideal = dcg_at_k(sorted(relevance_scores, reverse=True), k=k)
    if ideal == 0:
        return 0.0
    return actual / ideal


def f1_at_k(precision, recall):
    """Safe F1-score calculation for Precision@K and Recall@K."""
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


test_goals = [
    "short beginner python course",
    "beginner sql for data analysis",
    "learn data analysis for internship",
    "tableau course for dashboards",
    "power bi for beginners",
    "advanced machine learning course",
    "deep learning specialization",
    "free cybersecurity beginner course",
    "network security course",
    "cloud fundamentals aws",
    "azure beginner course",
    "project management certificate",
    "agile scrum course",
    "excel for business analytics",
    "statistics for data science",
    "nlp course with python",
    "react web development",
    "flutter mobile development",
    "java programming beginner",
    "c++ algorithms course",
    "hands-on data analytics project",
    "capstone machine learning course",
    "quick course on business intelligence",
    "professional certificate in cyber security",
]

evaluation_rows = []
summary_rows = []
binary_relevance_threshold = 1
top_k_eval = 15
fetch_k_eval = 50

for goal in test_goals:
    candidate_pool = rerank_results(
        retrieve_courses(goal, top_k=fetch_k_eval, fetch_k=fetch_k_eval),
        query=goal
    ).head(fetch_k_eval).reset_index(drop=True)

    graded_scores = []
    for _, row in candidate_pool.iterrows():
        rel_score, notes = evaluate_recommendation(goal, row)
        graded_scores.append(rel_score)
        evaluation_rows.append({
            "Goal": goal,
            "Course": row.get(title_col, "N/A"),
            "Rank": len(graded_scores),
            "Relevance Score (0-3)": rel_score,
            "Binary Relevant": int(rel_score >= binary_relevance_threshold),
            "Notes": notes,
            "Final Score": row.get("final_score", None),
        })

    top_scores = graded_scores[:top_k_eval]
    total_relevant_in_pool = sum(1 for s in graded_scores if s >= binary_relevance_threshold)
    relevant_in_top_k = sum(1 for s in top_scores if s >= binary_relevance_threshold)

    precision_at_k = relevant_in_top_k / top_k_eval
    recall_at_k = (relevant_in_top_k / total_relevant_in_pool) if total_relevant_in_pool > 0 else 0.0
    f1_score_at_k = f1_at_k(precision_at_k, recall_at_k)
    hit_rate_at_k = 1 if relevant_in_top_k > 0 else 0
    ndcg_k = ndcg_at_k(top_scores, k=top_k_eval)

    summary_rows.append({
        "Goal": goal,
        "Relevant in Pool": total_relevant_in_pool,
        f"Relevant in Top{top_k_eval}": relevant_in_top_k,
        f"Average Relevance@{top_k_eval}": float(np.mean(top_scores)) if top_scores else 0.0,
        f"Best Relevance@{top_k_eval}": max(top_scores) if top_scores else 0,
        f"Precision@{top_k_eval}": precision_at_k,
        f"Recall@{top_k_eval}": recall_at_k,
        f"F1-score@{top_k_eval}": f1_score_at_k,
        f"HitRate@{top_k_eval}": hit_rate_at_k,
        f"nDCG@{top_k_eval}": ndcg_k,
    })

evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df.head(20))


/tmp/ipykernel_1710/1068874826.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  results_df["duration_hours"].fillna(10**9) <= constraints["max_hours"]
/tmp/ipykernel_1710/1068874826.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  results_df["duration_hours"].fillna(10**9) <= constraints["max_hours"]


,Goal,Course,Rank,Relevance Score (0-3),Binary Relevant,Notes,Final Score
0,short beginner python course,Create Your First Python Program,1,3,1,Topic match; Constraint match; Practical signal,0.970935
1,short beginner python course,Python Basics,2,2,1,Topic match; Constraint match,0.893977
2,short beginner python course,Programming for Everybody (Getting Started with Python),3,1,1,Topic match,0.826539
3,short beginner python course,Python Programming Essentials,4,3,1,Topic match; Constraint match; Practical signal,0.779839
4,short beginner python course,Introduction to Python,5,3,1,Topic match; Constraint match; Practical signal,0.754867
5,short beginner python course,"Python Functions, Files, and Dictionaries",6,2,1,Topic match; Practical signal,0.721188
6,short beginner python course,Get Started with Python,7,1,1,Topic match,0.689682
7,short beginner python course,Create Your First Web App with Python and Flask,8,3,1,Topic match; Constraint match; Practical signal,0.689340
8,short beginner python course,Python Programming: A Concise Introduction,9,1,1,Topic match,0.665355
9,short beginner python course,Learn to Program: The Fundamentals,10,1,1,Topic match,0.654427


In [15]:

summary = pd.DataFrame(summary_rows)

precision_col = f"Precision@{top_k_eval}"
recall_col = f"Recall@{top_k_eval}"
f1_col = f"F1-score@{top_k_eval}"
hitrate_col = f"HitRate@{top_k_eval}"
ndcg_col = f"nDCG@{top_k_eval}"

display(summary.sort_values([f1_col, precision_col, recall_col, ndcg_col], ascending=False))

print(f"Overall Precision@{top_k_eval}:", round(summary[precision_col].mean(), 3))
print(f"Overall Recall@{top_k_eval}:", round(summary[recall_col].mean(), 3))
print(f"Overall F1-score@{top_k_eval}:", round(summary[f1_col].mean(), 3))
print(f"Overall HitRate@{top_k_eval}:", round(summary[hitrate_col].mean(), 3))
print(f"Overall nDCG@{top_k_eval}:", round(summary[ndcg_col].mean(), 3))


,Goal,Relevant in Pool,Relevant in Top15,Average Relevance@15,Best Relevance@15,Precision@15,Recall@15,F1-score@15,HitRate@15,nDCG@15
3,tableau course for dashboards,27,15,2.400000,3,1.000000,0.555556,0.714286,1,0.858615
16,react web development,29,15,2.466667,3,1.000000,0.517241,0.681818,1,0.947666
19,c++ algorithms course,40,15,2.333333,3,1.000000,0.375000,0.545455,1,0.874451
23,professional certificate in cyber security,48,15,1.266667,2,1.000000,0.312500,0.476190,1,0.745049
11,project management certificate,50,15,2.000000,2,1.000000,0.300000,0.461538,1,1.000000
12,agile scrum course,50,15,3.000000,3,1.000000,0.300000,0.461538,1,1.000000
21,capstone machine learning course,50,15,3.000000,3,1.000000,0.300000,0.461538,1,1.000000
20,hands-on data analytics project,50,15,2.800000,3,1.000000,0.300000,0.461538,1,0.992661
18,java programming beginner,50,15,2.733333,3,1.000000,0.300000,0.461538,1,0.956459
1,beginner sql for data analysis,50,15,2.266667,3,1.000000,0.300000,0.461538,1,0.943715


Overall Precision@15: 0.933
Overall Recall@15: 0.382
Overall F1-score@15: 0.474
Overall HitRate@15: 1.0
Overall nDCG@15: 0.898


In [16]:

try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception as e:
    print("This upload helper only works in Google Colab.", e)


Saving .env to .env
Uploaded files: ['.env']


In [17]:
from dotenv import load_dotenv
import os

env_loaded = False
for env_path in ["/content/.env", ".env"]:
    if os.path.exists(env_path):
        load_dotenv(env_path, override=True)
        print(f"Loaded environment file: {env_path}")
        env_loaded = True
        break

if not env_loaded:
    print("No .env file found. In Colab, upload .env to the Files panel or run: files.upload()")

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if api_key:
    os.environ["GEMINI_API_KEY"] = api_key
    print("Gemini API key loaded from .env: True")
else:
    print("Gemini API key loaded: False")


Loaded environment file: /content/.env
Gemini API key loaded from .env: True


In [18]:

def format_duration_label(hours):
    if pd.isna(hours) or hours is None:
        return "Unknown duration"
    hours = float(hours)
    if hours < 10:
        return f"{hours:.0f} hours"
    if hours < 40:
        weeks = max(1, round(hours / 8))
        return f"about {weeks} week(s)"
    if hours < 160:
        weeks = max(1, round(hours / 35))
        return f"about {weeks} week(s)"
    months = max(1, round(hours / 160))
    return f"about {months} month(s)"

def build_context(retrieved_df, top_k = 15):
    lines = []
    for _, row in retrieved_df.head(top_k).iterrows():
        title = safe_str(row.get(title_col, "N/A")) if title_col else "N/A"
        provider = safe_str(row.get(provider_col, "N/A")) if provider_col else "N/A"
        skills = safe_str(row.get(skills_col, "N/A")) if skills_col else "N/A"
        rating = safe_str(row.get(rating_col, "N/A")) if rating_col else "N/A"
        reviews = safe_str(row.get(reviews_col, "N/A")) if reviews_col else "N/A"
        level_value = safe_str(row.get("level", "N/A"))
        credential = safe_str(row.get("certificate_type", "N/A"))
        duration_label = format_duration_label(row.get("duration_hours", None))
        lines.append(
            f"Course {len(lines)+1}:\n"
            f"Title: {title}\n"
            f"Organization: {provider}\n"
            f"Level: {level_value}\n"
            f"Estimated duration: {duration_label}\n"
            f"Credential: {credential}\n"
            f"Skills: {skills}\n"
            f"Rating: {rating} | Reviews: {reviews}\n"
        )
    return "\n".join(lines)

def get_gemini_client():
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key or api_key == "YOUR_GEMINI_API_KEY":
        return None
    if genai is None:
        return None
    return genai.Client(api_key=api_key)

def generate_with_llm(prompt: str, model_name="gemini-2.5-flash", max_attempts=4) -> str:
    import time
    client = get_gemini_client()
    if client is None:
        raise RuntimeError("Gemini client is unavailable. Set GEMINI_API_KEY and ensure google-genai is installed.")

    last_error = None
    for attempt in range(max_attempts):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )
            return response.text
        except Exception as error:
            last_error = error
            message = str(error).lower()
            if "503" not in message and "unavailable" not in message and "high demand" not in message:
                raise error
            if attempt < max_attempts - 1:
                time.sleep(3 * (attempt + 1))
    raise last_error

def llm_generate_answer(goal_text, retrieved_df):
    context = build_context(retrieved_df, top_k=min(12, len(retrieved_df)))
    prompt = f"""
You are a learning roadmap generator.

User goal:
{goal_text}

You MUST use ONLY the courses listed below.
Do NOT invent new courses.
Do NOT add courses that are not in the provided context.

Create a concise roadmap-style answer.

Requirements:
1. Start with one short intro sentence.
2. Organize the answer into stages or months.
3. Order the courses from foundational to more advanced.
4. For each stage, include:
   - the course name
   - why this course belongs in this stage
   - the main skills the learner gains
5. Keep the answer concise, practical, and grounded in the provided courses only.
6. If the goal mentions a timeline, fit the roadmap into that timeline as reasonably as possible.

Courses:
{context}
"""
    return generate_with_llm(prompt)

def rag_recommend(goal_text, top_k = 15, fetch_k=50):
    ranked = recommend(goal_text, top_k=top_k, fetch_k=fetch_k)
    answer = llm_generate_answer(goal_text, ranked)

    print("\n" + "=" * 80)
    print("FINAL LLM ROADMAP")
    print("=" * 80)
    print(answer)
    print("=" * 80)

    cols = [c for c in [title_col, provider_col, skills_col, rating_col, reviews_col, "final_score"] if c]
    display(ranked[cols])
    return ranked


In [19]:

print("API key exists:", bool(os.getenv("GEMINI_API_KEY")) and os.getenv("GEMINI_API_KEY") != "YOUR_GEMINI_API_KEY")
client = get_gemini_client()
print("Client object:", client)


API key exists: True
Client object: <google.genai.client.Client object at 0x7a82b73fa360>


In [20]:
rag_recommend("learn data analysis and get me a 6-month roadmap", top_k = 15, fetch_k=50)

GOAL: learn data analysis and get me a 6-month roadmap
1) Your Guide to Learn Data Analysis
   Provider: almentor | Level: Beginner | Rating: 0.0
   Credential: 
   Why it matches: Matches skills/topics: data analysis; Aligned domain: data analysis
   Reference: course_id=1598 | url=https://www.almentor.net/en/courses/Your-Guide-to-Learn-Data-Analysis
----------------------------------------------------------------------------------------------------
2) Business Statistics and Analysis Capstone
   Provider: Rice University | Level: Beginner | Rating: 4.7
   Credential: 
   Why it matches: Matches skills/topics: data analysis; Aligned domain: data analysis; Strong rating (4.7)
   Reference: course_id=1052 | url=https://www.coursera.org/learn/business-statistics-analysis-capstone
----------------------------------------------------------------------------------------------------
3) Data Analysis with Python
   Provider: IBM | Level: Unknown | Rating: 4.6
   Credential: 
   Why it matches

,course_title,provider,skills,rating,reviews,final_score
0,Your Guide to Learn Data Analysis,almentor,data_analysis|business_intelligence There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this c...,0.0,0.0,0.565508
1,Business Statistics and Analysis Capstone,Rice University,statistical analysis microsoft excel business analytics regression analysis general statistics data analysis regression analytics analysis business analysis data-science data-analysis,4.7,0.0,0.529534
2,Data Analysis with Python,IBM,data model regression python programming regression analysis exploratory data analysis analysis computer programming modeling data analysis linearity data-science data-analysis,4.6,0.0,0.514190
3,Data Science Fundamentals for Data Analysts,Databricks,"data analysis, data science, machine learning",4.2,0.0,0.502294
4,Data Analytics for Lean Six Sigma,University of Amsterdam,minitab data analysis general statistics statistical analysis six sigma analysis regression lean six sigma trigonometric integral regression analysis data-science data-analysis,4.8,0.0,0.488122
5,The Data Scientist s Toolbox,Johns Hopkins University,r programming analysis git (software) data analysis github computer programming version control rstudio big data software data-science data-analysis,4.5,0.0,0.487575
6,Introduction to Data Analysis,kayfa,data_analysis,0.0,0.0,0.486650
7,Analyze Data to Answer Questions,Google,"data analysis, data management, microsoft excel, sql, spreadsheet software",4.6,0.0,0.471422
8,Data Science at Scale - Capstone Project,University of Washington,entropy (information theory) information engineering r programming communication general statistics problem solving feature engineering data analysis python programming analysis data-science data-...,3.8,0.0,0.458056
9,Exploratory Data Analysis,Coursera Project Network,,0.0,0.0,0.455380


,course_title,provider,level,rating,reviews,url,description,skills,track,prerequisites,...,doc_text,semantic_score,filter_applied,semantic_score_norm,rating_score_norm,review_score_norm,keyword_match_norm,constraint_match_norm,quality_weight_norm,final_score
0,Your Guide to Learn Data Analysis,almentor,Beginner,0.0,0.0,https://www.almentor.net/en/courses/Your-Guide-to-Learn-Data-Analysis,"Taught by Nermin Elgrawany. Having chunks of data is one thing, and using that data to empower strategic business decisions is a whole another story. Simply because, if you don’t know how to explo...",data_analysis|business_intelligence There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this c...,data_analysis|business_intelligence,There are no requirements for this course. Your interest in the topic and your commitment to learning are all you need to achieve the utmost benefit from this course.,...,"Title: Your Guide to Learn Data Analysis\nDescription: Taught by Nermin Elgrawany. Having chunks of data is one thing, and using that data to empower strategic business decisions is a whole anothe...",0.676290,True,1.000000,0.000000,0.0,0.0,0.0,0.82,0.565508
1,Business Statistics and Analysis Capstone,Rice University,Beginner,4.7,0.0,https://www.coursera.org/learn/business-statistics-analysis-capstone,"The Business Statistics and Analysis Capstone is an opportunity to apply various skills developed across the four courses in the specialization to a real life data. The Capstone, in collaboration ...",statistical analysis microsoft excel business analytics regression analysis general statistics data analysis regression analytics analysis business analysis data-science data-analysis,semi_it,,...,Title: Business Statistics and Analysis Capstone\nDescription: The Business Statistics and Analysis Capstone is an opportunity to apply various skills developed across the four courses in the spec...,0.631019,True,0.656913,0.979167,0.0,0.0,0.0,0.70,0.529534
2,Data Analysis with Python,IBM,Unknown,4.6,0.0,https://www.coursera.org/learn/data-analysis-with-python,"Learn how to analyze data using Python. This course will take you from the basics of Python to exploring many different types of data. You will learn how to prepare data for analysis, perform simp...",data model regression python programming regression analysis exploratory data analysis analysis computer programming modeling data analysis linearity data-science data-analysis,core_it,,...,Title: Data Analysis with Python\nDescription: Learn how to analyze data using Python. This course will take you from the basics of Python to exploring many different types of data. You will learn...,0.606045,True,0.467645,0.958333,0.0,0.0,0.0,1.00,0.514190
3,Data Science Fundamentals for Data Analysts,Databricks,Intermediate,4.2,0.0,,"Data Science Fundamentals for Data Analysts covers data analysis, data science, machine learning","data analysis, data science, machine learning",core_it,,...,"Title: Data Science Fundamentals for Data Analysts\nDescription: Data Science Fundamentals for Data Analysts covers data analysis, data science, machine learning\nSkills: data analysis, data scien...",0.613157,True,0.521540,0.875000,0.0,0.0,0.0,0.90,0.502294
4,Data Analytics for Lean Six Sigma,University of Amsterdam,Advanced,4.8,0.0,https://www.coursera.org/learn/data-analytics-for-lean-six-sigma,Welcome to this course on Data Analytics for Lean Six Sigma. In this course you will learn data analytics techniques that are typically useful within Lean Six Sigma improvement projects. At the en...,minitab data analysis general statistics statistical analysis six sigma analysis regression lean six sigma trigonometric integral regression analysis data-science data-analysis,core_it,,...,Title: Data Analytics for Lean Six Sigma\nDescription: Welcome to this course on Data Analytics for Lean Six Sigma. In this course you will learn data 